# Day 02: The Extended Kalman Filter (EKF) - Nonlinear Kinematics & Radar Tracking
**State Estimation and Localization for Self-Driving Cars**

### 🎯 Learning Objectives:
1. Master the **First-Order Taylor Series Linearization** of nonlinear motion $\mathbf{f}(\mathbf{x}, \mathbf{u})$ and sensor observation $\mathbf{h}(\mathbf{x})$ functions.
2. Understand the systematic derivation of state transition Jacobians $\mathbf{F}_{k-1}, \mathbf{L}_{k-1}$ and measurement Jacobians $\mathbf{H}_k, \mathbf{M}_k$.
3. Implement the generic multi-dimensional `ExtendedKalmanFilter` class following standard academic formulations.
4. Apply EKF to 2D automotive radar tracking with range $r$ and azimuth angle $\phi$ in polar coordinates.
5. Evaluate filter statistical consistency using the Normalized Estimation Error Squared (NEES) and $\chi^2$ hypothesis testing.
6. Build interactive multi-panel Plotly diagnostic dashboards.

---
## 1. Environment Setup

In [26]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2

# Set random seed for reproducible stochastic simulations
np.random.seed(42)
print('Environment ready: NumPy, SciPy, and Plotly loaded.')

Environment ready: NumPy, SciPy, and Plotly loaded.


---
## 2. Mathematical Foundation: Discrete Extended Kalman Filter

In autonomous systems, kinematics and sensor measurements are fundamentally **nonlinear**:

$$\mathbf{x}_k = \mathbf{f}(\mathbf{x}_{k-1}, \mathbf{u}_{k-1}) + \mathbf{w}_{k-1}, \quad \mathbf{w}_{k-1} \sim \mathcal{N}(\mathbf{0}, \mathbf{Q}_{k-1})$$
$$\mathbf{y}_k = \mathbf{h}(\mathbf{x}_k) + \mathbf{v}_k, \quad \mathbf{v}_k \sim \mathcal{N}(\mathbf{0}, \mathbf{R}_k)$$

The Extended Kalman Filter linearizes $\mathbf{f}$ and $\mathbf{h}$ locally around the current state estimate via 1st-order Taylor expansions:

$$\mathbf{F}_{k-1} = \left. \frac{\partial \mathbf{f}}{\partial \mathbf{x}} \right|_{\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1}}, \quad \mathbf{L}_{k-1} = \left. \frac{\partial \mathbf{f}}{\partial \mathbf{w}} \right|_{\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1}}$$
$$\mathbf{H}_k = \left. \frac{\partial \mathbf{h}}{\partial \mathbf{x}} \right|_{\check{\mathbf{x}}_k}, \quad \mathbf{M}_k = \left. \frac{\partial \mathbf{h}}{\partial \mathbf{v}} \right|_{\check{\mathbf{x}}_k}$$

### 🔁 Discrete EKF Recursive Algorithm:

| Step | Mathematical Formula | Academic Definition |
| :--- | :--- | :--- |
| **1. State Prediction** | $\check{\mathbf{x}}_k = \mathbf{f}(\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1})$ | Propagate state through full nonlinear kinematic dynamics |
| **2. Covariance Prediction** | $\check{\mathbf{P}}_k = \mathbf{F}_{k-1}\hat{\mathbf{P}}_{k-1}\mathbf{F}_{k-1}^T + \mathbf{L}_{k-1}\mathbf{Q}_{k-1}\mathbf{L}_{k-1}^T$ | Linearized uncertainty expansion |
| **3. Innovation Residual** | $\mathbf{\nu}_k = \mathbf{y}_k - \mathbf{h}(\check{\mathbf{x}}_k)$ | Discrepancy between actual sensor data and predicted measurement |
| **4. Innovation Covariance** | $\mathbf{S}_k = \mathbf{H}_k\check{\mathbf{P}}_k\mathbf{H}_k^T + \mathbf{M}_k\mathbf{R}_k\mathbf{M}_k^T$ | Total uncertainty projected in measurement space |
| **5. Kalman Gain** | $\mathbf{K}_k = \check{\mathbf{P}}_k\mathbf{H}_k^T \mathbf{S}_k^{-1}$ | Optimal minimum mean squared error (MMSE) gain |
| **6. State Correction** | $\hat{\mathbf{x}}_k = \check{\mathbf{x}}_k + \mathbf{K}_k \mathbf{\nu}_k$ | A posteriori optimal state estimate |
| **7. Covariance Correction** | $\hat{\mathbf{P}}_k = (\mathbf{I} - \mathbf{K}_k\mathbf{H}_k)\check{\mathbf{P}}_k(\mathbf{I} - \mathbf{K}_k\mathbf{H}_k)^T + \mathbf{K}_k\mathbf{R}_{\text{eff}}\mathbf{K}_k^T$ | Joseph form for numerical stability & positive semi-definiteness |

---
## 3. Implementing the `ExtendedKalmanFilter` Class

### 📝 Exercise 1: Implement the Generic EKF Class

In [19]:
def wrap_angle(angle):
    """Wraps an angle or array of angles to the interval [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class ExtendedKalmanFilter:
    """Generic Multi-Dimensional Extended Kalman Filter (EKF)."""
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray):
        self.x = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.n = self.x.shape[0]
        
        self.latest_innovation = None
        self.latest_innovation_cov = None
        self.latest_gain = None
        
    def predict(self, f_func, F_jac: np.ndarray, Q: np.ndarray, 
                u: np.ndarray = None, L_jac: np.ndarray = None):
        # -------------------------------------------------------------------------
        # 1. State propagation: check_x = f(hat_x, u) (or f(hat_x) if u is None)
        # 2. Covariance propagation: check_P = F * hat_P * F^T + L * Q * L^T (L=I if None)
        # -------------------------------------------------------------------------
        if u is not None:
            self.x = f_func(self.x, u).reshape(-1, 1)
        else:
            self.x = f_func(self.x).reshape(-1, 1)
        if L_jac is None:
            L_jac = np.eye(self.n)
        self.P = F_jac @ self.P @ F_jac.T + L_jac @ Q @ L_jac.T
        return self.x.copy(), self.P.copy()

        
    def update(self, y: np.ndarray, h_func, H_jac: np.ndarray, R: np.ndarray, 
               M_jac: np.ndarray = None, angle_indices: list = None):
        # -------------------------------------------------------------------------
        # 1. Innovation: nu = y - h(check_x) (wrap angles if angle_indices specified)
        # 2. Innovation Covariance: S = H * check_P * H^T + M * R * M^T (M=I if None)
        # 3. Kalman Gain: K = check_P * H^T * inv(S)
        # 4. State Correction: hat_x = check_x + K * nu
        # 5. Covariance Correction: Joseph form (I - K*H) * check_P * (I - K*H)^T + K*R_eff*K^T
        # -------------------------------------------------------------------------
        y_vec = np.asarray(y, dtype=np.float64).reshape(-1, 1)
        m = y_vec.shape[0]
        if M_jac is None:
            M_jac = np.eye(m)

        # Innovation
        nu = y_vec - h_func(self.x).reshape(-1, 1)
        if angle_indices is not None:
            for idx in angle_indices:
                nu[idx, 0] = wrap_angle(nu[idx, 0])

        ## Covariancia
        R_eff = M_jac @ R @ M_jac.T
        S = H_jac @ self.P @ H_jac.T + R_eff ## Inovaci[on de la covarianza

        # Ganancia de Kalman
        K = self.P @ H_jac.T @ np.linalg.inv(S)

        # Correccion del estado
        self.x = self.x + K @ nu

        # Correccion de covarianza
        I_KH = np.eye(self.n) - K @ H_jac
        self.P = I_KH @ self.P @ I_KH.T + K @ R_eff @ K.T

        self.latest_innovation = nu
        self.latest_innovation_cov = S
        self.latest_gain = K

        return self.x.copy(), self.P.copy()


In [20]:
def wrap_angle(angle):
    """Wraps an angle or array of angles to the interval [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class ExtendedKalmanFilter:
    """Generic Multi-Dimensional Extended Kalman Filter (EKF).
    
    Keeps all filter methods completely generic, accepting arbitrary nonlinear functions
    f(x, u), h(x), Jacobians F, L, H, M, and noise covariance matrices Q, R.
    """
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray):
        self.x = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.n = self.x.shape[0]
        
        self.latest_innovation = None
        self.latest_innovation_cov = None
        self.latest_gain = None
        
    def predict(self, f_func, F_jac: np.ndarray, Q: np.ndarray, 
                u: np.ndarray = None, L_jac: np.ndarray = None):
        """Executes the EKF State and Covariance Prediction Step:
            check_x = f(hat_x, u)
            check_P = F * hat_P * F^T + L * Q * L^T
        """
        if u is not None:
            self.x = f_func(self.x, u).reshape(-1, 1)
        else:
            self.x = f_func(self.x).reshape(-1, 1)
            
        if L_jac is None:
            L_jac = np.eye(self.n)
            
        self.P = F_jac @ self.P @ F_jac.T + L_jac @ Q @ L_jac.T
        return self.x.copy(), self.P.copy()
        
    def update(self, y: np.ndarray, h_func, H_jac: np.ndarray, R: np.ndarray, 
               M_jac: np.ndarray = None, angle_indices: list = None):
        """Executes the EKF Measurement Correction Step:
            1. Innovation:            nu = y - h(check_x)
            2. Innovation Covariance: S  = H * check_P * H^T + M * R * M^T
            3. Kalman Gain:           K  = check_P * H^T * inv(S)
            4. Corrected State:       hat_x = check_x + K * nu
            5. Corrected Covariance:  hat_P = (I - K*H) * check_P * (I - K*H)^T + K * R_eff * K^T (Joseph form)
        """
        y_vec = np.asarray(y, dtype=np.float64).reshape(-1, 1)
        m = y_vec.shape[0]
        if M_jac is None:
            M_jac = np.eye(m)
            
        # 1. Innovation residual
        y_pred = h_func(self.x).reshape(-1, 1)
        nu = y_vec - y_pred
        if angle_indices is not None:
            for idx in angle_indices:
                nu[idx, 0] = wrap_angle(nu[idx, 0])
                
        # 2. Innovation Covariance
        R_eff = M_jac @ R @ M_jac.T
        S = H_jac @ self.P @ H_jac.T + R_eff
        
        # 3. Optimal Kalman Gain
        K = self.P @ H_jac.T @ np.linalg.inv(S)
        
        # 4. Posterior State Estimate
        self.x = self.x + K @ nu
        
        # 5. Posterior Covariance (Joseph form for numerical stability)
        I_KH = np.eye(self.n) - K @ H_jac
        self.P = I_KH @ self.P @ I_KH.T + K @ R_eff @ K.T
        self.P = 0.5 * (self.P + self.P.T)
        
        self.latest_innovation = nu
        self.latest_innovation_cov = S
        self.latest_gain = K
        
        return self.x.copy(), self.P.copy()

print('Generic ExtendedKalmanFilter class compiled successfully.')

Generic ExtendedKalmanFilter class compiled successfully.


### 🧪 Unit Test: Generic ExtendedKalmanFilter Sanity Check (with Control Input & Measurement Update)

In [21]:
# Sanity verification of generic ExtendedKalmanFilter class
dt_test = 0.1
f_test = lambda x, u=None: np.array([[x[0, 0] + dt_test * x[1, 0]], [x[1, 0]]])
F_test = np.array([[1.0, dt_test], [0.0, 1.0]])
Q_test = np.diag([0.01, 0.05])
ekf_test = ExtendedKalmanFilter(x0=np.array([10.0, 2.0]), P0=np.eye(2) * 4.0)

# 1. Test Prediction Step
x_pred, P_pred = ekf_test.predict(f_func=f_test, F_jac=F_test, Q=Q_test)
assert x_pred.shape == (2, 1), f"Expected shape (2, 1), got {x_pred.shape}"
assert np.isclose(x_pred[0, 0], 10.2), f"Expected pos 10.2, got {x_pred[0, 0]}"
assert np.isclose(x_pred[1, 0], 2.0), f"Expected vel 2.0, got {x_pred[1, 0]}"
assert P_pred.shape == (2, 2), f"Expected shape (2, 2), got {P_pred.shape}"
assert P_pred[0, 0] > 4.0, "Covariance must expand after prediction"

# 2. Test Measurement Update Step
h_test = lambda x: np.array([[x[0, 0]]])
H_test = np.array([[1.0, 0.0]])
R_test = np.array([[0.25]])
x_upd, P_upd = ekf_test.update(y=np.array([10.5]), h_func=h_test, H_jac=H_test, R=R_test)
assert x_upd.shape == (2, 1), f"Expected shape (2, 1), got {x_upd.shape}"
assert P_upd.shape == (2, 2), f"Expected shape (2, 2), got {P_upd.shape}"
assert P_upd[0, 0] < P_pred[0, 0], "Covariance must decrease after update"
assert np.all(np.linalg.eigvals(P_upd) > 0), "P must remain strictly positive definite!"
print('✅ ExtendedKalmanFilter generic class unit tests passed!')

✅ ExtendedKalmanFilter generic class unit tests passed!


In [22]:
def motion_model(x, u, dt):
    """Nonlinear kinematic state transition f(x, u)."""
    px,  py, v, theta = x.flatten()
    a, omega = u.flatten()
    px_next = px + v * np.cos(theta) * dt + 0.5 * a * dt**2 * np.cos(theta)
    py_next = py + v * np.sin(theta) * dt + 0.5 * a * dt**2 * np.sin(theta)
    v_next = v + a * dt
    theta_next = wrap_angle(theta + omega * dt)
    return np.array([px_next, py_next, v_next, theta_next]).reshape(-1, 1)

def get_F_jacobian(x, dt, u):
    """Computes 4x4 state transition Jacobian F = df/dx."""
    px, py, v, theta = x.flatten()
    a, omega = u.flatten()
    return np.array([
        [1.0, 0.0, np.cos(theta) * dt, -v * np.sin(theta) * dt - 0.5 * a * dt ** 2 * np.sin(theta)],
        [0.0, 1.0, np.sin(theta) * dt, v * np.cos(theta) * dt + 0.5 * a * dt ** 2 * np.cos(theta)],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ])

def measurement_model(x):
    """Computes range and bearing [r, phi]^T from Cartesian state."""
    px, py = x[0, 0], x[1, 0]
    r = np.sqrt(px ** 2 + py ** 2)
    phi = np.arctan2(py, px)
    return np.array([r, phi]).reshape(-1, 1)

def get_H_jacobian(x):
    """Computes 2x4 measurement Jacobian H."""
    px, py = x[0, 0], x[1, 0]
    r2 = max(px ** 2 + py ** 2, 1e-8)
    r = np.sqrt(r2)
    return np.array([
        [px / r, py / r, 0.0, 0.0],
        [-py / r2, px / r2, 0.0, 0.0]
    ])


### 🧪 Unit Test: 2D Polar Radar Kinematics & Jacobians Sanity Check

In [23]:
# Sanity verification of 2D Polar Radar Kinematics and Jacobians
x_test = np.array([[10.0], [0.0], [5.0], [0.0]])  # px=10, py=0, v=5, theta=0
u_test = np.array([[2.0], [0.5]])                  # a=2, omega=0.5
dt_test = 0.1

# 1. Test Motion Model
x_next = motion_model(x_test, u_test, dt_test)
assert x_next.shape == (4, 1), f"Expected shape (4, 1), got {x_next.shape}"
assert np.isclose(x_next[0, 0], 10.51), f"Expected px=10.5, got {x_next[0, 0]}"
assert np.isclose(x_next[1, 0], 0.0), f"Expected py=0.0, got {x_next[1, 0]}"
assert np.isclose(x_next[2, 0], 5.2), f"Expected v=5.2, got {x_next[2, 0]}"
assert np.isclose(x_next[3, 0], 0.05), f"Expected theta=0.05, got {x_next[3, 0]}"

# 2. Test F Jacobian
F_eval = get_F_jacobian(x_test, dt_test, u_test)
assert F_eval.shape == (4, 4), f"Expected shape (4, 4), got {F_eval.shape}"
assert np.isclose(F_eval[0, 2], dt_test), "df1/dv must equal cos(theta)*dt"
assert np.isclose(F_eval[1, 3], 5.0 * dt_test+ 0.5 * 2.0 * dt_test ** 2), "df2/dtheta must equal v*cos(theta)*dt"

# 3. Test Measurement Model (target at (3, 4) -> r=5, phi=atan2(4,3))
x_radar = np.array([[3.0], [4.0], [0.0], [0.0]])
y_radar = measurement_model(x_radar)
assert y_radar.shape == (2, 1), f"Expected shape (2, 1), got {y_radar.shape}"
assert np.isclose(y_radar[0, 0], 5.0), f"Expected range 5.0, got {y_radar[0, 0]}"
assert np.isclose(y_radar[1, 0], np.arctan2(4.0, 3.0)), f"Expected bearing {np.arctan2(4.0, 3.0)}, got {y_radar[1, 0]}"

# 4. Test H Jacobian
H_eval = get_H_jacobian(x_radar)
assert H_eval.shape == (2, 4), f"Expected shape (2, 4), got {H_eval.shape}"
assert np.isclose(H_eval[0, 0], 3.0 / 5.0), "dr/dpx should be px/r = 3/5"
assert np.isclose(H_eval[0, 1], 4.0 / 5.0), "dr/dpy should be py/r = 4/5"
assert np.isclose(H_eval[1, 0], -4.0 / 25.0), "dphi/dpx should be -py/r^2 = -4/25"
assert np.isclose(H_eval[1, 1], 3.0 / 25.0), "dphi/dpy should be px/r^2 = 3/25"
print('✅ 2D Radar Kinematics and Measurement Jacobians unit tests passed!')

✅ 2D Radar Kinematics and Measurement Jacobians unit tests passed!


---
## 5. Physical Formulation of $\mathbf{Q}$ and $\mathbf{R}$ Matrices

### 🔹 Process Noise Covariance $\mathbf{Q}$ (State $\mathbf{x} = [p_x, p_y, v, \theta]^T$):
* $\sigma_{p_x} = 0.05\text{ m}, \sigma_{p_y} = 0.05\text{ m}$: Models tire lateral slip and 1st-order discrete kinematic truncation over $\Delta t = 0.1\text{ s}$.
* $\sigma_v = 0.1\text{ m/s}$: Models engine throttle response lag, road slope variations, and braking jitter.
* $\sigma_\theta = 0.02\text{ rad} \approx 1.15^\circ$: Models steering backlash, crosswinds, and road bank angle.
$$\mathbf{Q} = \operatorname{diag}(0.05^2, 0.05^2, 0.1^2, 0.02^2)$$

### 🔹 Measurement Noise Covariance $\mathbf{R}$ (Radar $\mathbf{y} = [r, \phi]^T$):
* $\sigma_r = 0.5\text{ m}$: Automotive mmWave radar range ToF measurement precision.
* $\sigma_\phi = 1.0^\circ = 0.0175\text{ rad}$: Radar antenna array beam azimuth resolution.
$$\mathbf{R} = \operatorname{diag}(0.5^2, (\operatorname{deg2rad}(1.0))^2)$$

---
## 6. Simulation & Filter Execution

In [24]:
dt = 0.1
T_total = 60.0
N_steps = int(T_total / dt)
time = np.linspace(0, T_total, N_steps)

x_true_all = np.zeros((4, N_steps))
x_true = np.array([10.0, 5.0, 15.0, 0.0]).reshape(-1, 1)

Q = np.diag([0.05**2, 0.05**2, 0.1**2, 0.02**2])
R = np.diag([0.5**2, np.deg2rad(1.0)**2])

# Generate true kinematics and radar measurements
measurements = []
for k in range(N_steps):
    t = time[k]
    a_cmd = 0.5 * np.sin(0.1 * t)
    omega_cmd = 0.1 * np.cos(0.08 * t)
    u = np.array([a_cmd, omega_cmd]).reshape(-1, 1)
    w = np.random.multivariate_normal(np.zeros(4), Q).reshape(-1, 1)
    x_true = motion_model(x_true, u, dt) + w
    x_true[3, 0] = wrap_angle(x_true[3, 0])
    x_true_all[:, k] = x_true.flatten()
    v_noise = np.random.multivariate_normal(np.zeros(2), R).reshape(-1, 1)
    y = measurement_model(x_true) + v_noise
    y[1, 0] = wrap_angle(y[1, 0])
    measurements.append((u, y))

# Initialize Generic Extended Kalman Filter
x0_est = np.array([8.0, 3.0, 10.0, np.deg2rad(10.0)]).reshape(-1, 1)
P0_est = np.diag([5.0**2, 5.0**2, 5.0**2, np.deg2rad(20.0)**2])
ekf = ExtendedKalmanFilter(x0_est, P0_est)

x_est_all = np.zeros((4, N_steps))
P_diag_all = np.zeros((4, N_steps))
nees_all = np.zeros(N_steps)

for k in range(N_steps):
    u, y = measurements[k]
    # 1. Prediction
    F = get_F_jacobian(ekf.x, dt,u)
    ekf.predict(lambda x, u_in: motion_model(x, u_in, dt), F, Q, u=u)
    
    # 2. Measurement Update
    H = get_H_jacobian(ekf.x)
    ekf.update(y, measurement_model, H, R, angle_indices=[1])
    
    # Store trajectory and metrics
    x_est_all[:, k] = ekf.x.flatten()
    P_diag_all[:, k] = np.diag(ekf.P)
    err = x_true_all[:, k:k+1] - ekf.x
    err[3, 0] = wrap_angle(err[3, 0])
    nees_all[k] = (err.T @ np.linalg.inv(ekf.P) @ err).item()

rmse_pos = np.sqrt(np.mean((x_true_all[0, :] - x_est_all[0, :])**2 + (x_true_all[1, :] - x_est_all[1, :])**2))
print(f'EKF execution complete. Position RMSE: {rmse_pos:.3f} m | Mean NEES: {np.mean(nees_all):.2f}')

EKF execution complete. Position RMSE: 3.087 m | Mean NEES: 4.76


In [25]:
---
## 7. Plotly Interactive Visualizations

SyntaxError: invalid syntax (679168066.py, line 1)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '2D Vehicle Trajectory Tracking',
        'Position Error vs. 3-Sigma Bound',
        'Heading Error vs. 3-Sigma Bounds',
        'Filter Consistency: NEES (dim=4)'
    )
)

radar_x = [m[1][0, 0] * np.cos(m[1][1, 0]) for m in measurements]
radar_y = [m[1][0, 0] * np.sin(m[1][1, 0]) for m in measurements]

fig.add_trace(go.Scatter(x=x_true_all[0, :], y=x_true_all[1, :], mode='lines', name='Ground Truth', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=radar_x[::4], y=radar_y[::4], mode='markers', name='Radar Points', marker=dict(color='red', size=4, opacity=0.4)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_est_all[0, :], y=x_est_all[1, :], mode='lines', name='EKF Estimate', line=dict(color='blue', width=2, dash='dash')), row=1, col=1)

pos_err = np.sqrt((x_true_all[0, :] - x_est_all[0, :])**2 + (x_true_all[1, :] - x_est_all[1, :])**2)
sigma_pos = 3.0 * np.sqrt(P_diag_all[0, :] + P_diag_all[1, :])
fig.add_trace(go.Scatter(x=time, y=pos_err, mode='lines', name='Position Error [m]', line=dict(color='blue', width=1.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=time, y=sigma_pos, mode='lines', name='+3-Sigma Bound [m]', line=dict(color='red', dash='dot', width=1.5)), row=1, col=2)

heading_err = wrap_angle(x_true_all[3, :] - x_est_all[3, :])
sigma_heading = 3.0 * np.sqrt(P_diag_all[3, :])
fig.add_trace(go.Scatter(x=time, y=np.rad2deg(heading_err), mode='lines', name='Heading Error [deg]', line=dict(color='green', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=np.rad2deg(sigma_heading), mode='lines', name='+3-Sigma [deg]', line=dict(color='red', dash='dot', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=-np.rad2deg(sigma_heading), mode='lines', name='-3-Sigma [deg]', line=dict(color='red', dash='dot', width=1.5), showlegend=False), row=2, col=1)

chi2_low, chi2_high = chi2.interval(0.95, df=4)
fig.add_trace(go.Scatter(x=time, y=nees_all, mode='lines', name='NEES', line=dict(color='black', width=1.2)), row=2, col=2)
fig.add_trace(go.Scatter(x=time, y=[chi2_high]*N_steps, mode='lines', name='95% Upper (9.49)', line=dict(color='red', dash='dash')), row=2, col=2)
fig.add_trace(go.Scatter(x=time, y=[chi2_low]*N_steps, mode='lines', name='95% Lower (0.48)', line=dict(color='orange', dash='dash')), row=2, col=2)

fig.update_layout(title_text='Day 2 EKF: 2D Radar Tracking & Performance Diagnostics', template='plotly_white', height=750, width=1050)
fig.show()